## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

import json

from pathlib import Path

import pickle

from functions.json_functions import *

## Importing data

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
json_files_dict = {}

for path in sorted(Path('../raw_data-originals/dati_eumaco_con_pseudo').glob('*.json')):
    try:
        filename = path.stem
        json_content = json.loads(path.read_text()) # Closes automatically the file
        json_files_dict[filename] = json_content
    except json.JSONDecodeError:
        print(f"Skipping invalid JSON: {path}")

# List of all the json files names
json_filenames = list(json_files_dict)

## Make a pretty-print version of all the jsons

In [ ]:
# input_dir = '../raw_data-originals/dati_eumaco_con_pseudo'
# output_dir = '../raw_data/jsons-pp'

# pretty_print_folder(input_dir, output_dir)

### Put specific values of the json in a pandas df

In [ ]:
external_fields = [
    'patient_cf' ,
    'assignment_id' ,
    'archivied' ,
    'completed' ,

    'assignment:id' ,
    'assignment:start_at' ,
    'assignment:end_at' ,
    'assignment:active' ,
    'assignment:created_at' ,
    'assignment:updated_at' ,
    'assignment:hub_sn' ,
    'assignment:completed' ,
]

instance_ext_fields = [
    'id' ,
    'type' ,
    'report' ,
    'created_at' ,
    'updated_at' ,
    'registration_time' ,
    'device_sn' ,

# transmission_data
    'transmission_data:device' ,
    'transmission_data:serialnumber' ,
    'transmission_data:timestamp' ,
# battery
    'transmission_data:battery:value' ,
    'transmission_data:battery:unit' ,
# spo2
    'transmission_data:spo2:value' ,
    'transmission_data:spo2:unit' ,
# heartrate
    'transmission_data:heartrate:value' ,
    'transmission_data:heartrate:unit' ,
]

In [ ]:
data = json_to_df(json_files_dict, json_filenames,
    just_external = True,
    transm_field = 'SpO2Transmission',
    external_json_fields = external_fields,
    transm_inst_fields = instance_ext_fields
)
df = pd.DataFrame(data)

In [ ]:
df.info()

## Analysis and overview

### Check if the "external" fields except the `transmissions` have the consistent values

#### booleans

##### assignment_ids

In [ ]:
column1 = 'assignment_id'
column2 = 'assignment:id'
print(columns_consistency(df, column1, column2, return_number=True))

##### archived/completed

In [ ]:
column1 = 'archivied'
column2 = 'completed'

print(columns_consistency(df, column2, column1, return_number=True))

for json in columns_consistency(df, column2, column1).index:
    print(f'{json_filenames[json]}.json')

##### completed/assignment:completed

In [ ]:
column1 = 'completed'
column2 = 'assignment:completed'

print(columns_consistency(df, column2, column1, return_number=True))

##### assignment:active/assignment:completed

In [ ]:
column1 = 'assignment:active'
column2 = 'assignment:completed'

print(columns_consistency(df, column2, column1, return_number=True))

for json in columns_consistency(df, column2, column1).index:
    print(f'{json_filenames[json]}.json')

##### archivied/assignment:completed

In [ ]:
column1 = 'archivied'
column2 = 'assignment:completed'

print(columns_consistency(df, column2, column1, return_number=True))

for json in columns_consistency(df, column2, column1).index:
    print(f'{json_filenames[json]}.json')

## Parsing and reformatting

Shows a sample of the field where the useful data is stored 

In [ ]:
json_files_dict[json_filenames[0]]['transmissions']['SpO2Transmission'][0][0]

### Put all the `patient_cf` fields in a list

In [ ]:
# ids = []

# for json in json_files:
#     ids.append(json['patient_cf'])

In [ ]:
# ser = pd.Series(ids)
# ser

## Export

In [ ]:
# # Pickle
# with open('../raw_data/pickles/4-eumaco-df.pickle', 'wb') as file:
#     pickle.dump(df, file, protocol=pickle.HIGHEST_PROTOCOL)